# Font Identifier - Training Notebook

## Session Setup
Run the cell below first to initialize your environment, mount Drive, and pull the latest code.

In [ ]:
# ⚠️ RUN THIS FIRST after every reconnect

# ── 1. Mount Drive ────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# ── 2. Pull latest code from GitHub ──────────────────────
import os

REPO_DIR  = '/content/fontidentifier'
DRIVE_DIR = '/content/drive/MyDrive/font-identifier'

if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull   # update if already cloned
else:
    !git clone https://github.com/jtheanonymous1707-wq/fontidentifier.git {REPO_DIR}

# ── 3. Add model folder to Python path ───────────────────
import sys
MODEL_DIR = f'{REPO_DIR}/model'
if MODEL_DIR not in sys.path:
    sys.path.insert(0, MODEL_DIR)

# ── 4. Install dependencies ───────────────────────────────
!pip install -q timm supabase python-dotenv

# ── 5. Verify GPU ─────────────────────────────────────────
import torch
try:
    print(f"GPU: {torch.cuda.get_device_name(0)}")
except:
    print("GPU not found - ensure you are using a GPU runtime!")
print(f"sys.path includes model: {MODEL_DIR in sys.path}")

# ── 6. Test import ────────────────────────────────────────
import train
print(f"train.py loaded ✅")
print(f"FontNet available: {hasattr(train, 'FontNet')}")

## Step 5 — Set Environment Variables

In [ ]:
import os
os.environ['GOOGLE_FONTS_API_KEY'] = 'your_api_key'
os.environ['SUPABASE_URL']         = 'your_supabase_url'
os.environ['SUPABASE_SERVICE_KEY'] = 'your_service_key'
print("Secrets set (locally in memory).")

## Step 6 & 7 — Extract Dataset from Drive
Instead of downloading and generating images in every session, we unzip the pre-generated files from Drive.

In [ ]:
import zipfile, os, glob, shutil

print("🚀 Starting extraction from Drive...")

# Check for both possible filenames to be robust (fix for FileNotFoundError)
possible_zips = [f'{DRIVE_DIR}/font_data.zip', f'{DRIVE_DIR}/dataset.zip']
DATA_ZIP = next((z for z in possible_zips if os.path.exists(z)), None)
DATASET_DIR = '/content/data/dataset'

if DATA_ZIP:
    print(f"📦 Found archive: {DATA_ZIP} ({os.path.getsize(DATA_ZIP)/1024/1024:.2f} MB)")
    
    # Only extract if not already there or empty
    if not os.path.exists(DATASET_DIR) or len(glob.glob(f'{DATASET_DIR}/*')) == 0:
        print("⌛ Unzipping...")
        if os.path.exists('/content/data'): shutil.rmtree('/content/data')
        with zipfile.ZipFile(DATA_ZIP, 'r') as z:
            z.extractall('/content/')
        
        # Normalize if zipped directly
        if os.path.exists('/content/dataset'):
            os.makedirs('/content/data', exist_ok=True)
            shutil.move('/content/dataset', '/content/data/dataset')
    else:
        print("Dataset already extracted.")

    # ── Step 2: Purge Empty Folders (fix4.md) ────────────────
    print("Scanning for empty class folders...")
    removed = 0
    for class_dir in os.listdir(DATASET_DIR):
        full_path = os.path.join(DATASET_DIR, class_dir)
        if os.path.isdir(full_path) and len(glob.glob(f'{full_path}/*.png')) == 0:
            shutil.rmtree(full_path)
            removed += 1
    
    total_classes = len(os.listdir(DATASET_DIR))
    total_images  = len(glob.glob(f'{DATASET_DIR}/*/*.png'))
    print(f"✅ Extraction Complete.")
    print(f"✅ Removed {removed} empty folders.")
    print(f"✅ Ready: {total_images} images across {total_classes} classes.")
else:
    print(f"❌ Error: No dataset zip found in {DRIVE_DIR}!")
    print(f"Expected one of: {possible_zips}")

## Step 8 — Start Training
This loop includes the explicit auto-resume logic.

In [ ]:
import train

DRIVE_DIR = '/content/drive/MyDrive/font-identifier'

train.train(
    dataset_dir     = '/content/data/dataset',
    save_dir        = DRIVE_DIR,
    checkpoint_path = f'{DRIVE_DIR}/checkpoints/latest.pt',
    best_model_path = f'{DRIVE_DIR}/checkpoints/best_model.pt',
    epochs          = 60,
    batch_size      = 64,
)

## Step 9 — Export Final Model

In [ ]:
import os, torch, sys
sys.path.insert(0, '/content/fontidentifier/model')
import train

DRIVE_DIR       = '/content/drive/MyDrive/font-identifier'
BEST_MODEL_PATH = f'{DRIVE_DIR}/checkpoints/best_model.pt'

device = torch.device('cuda')

# Load best checkpoint
best_ckpt   = torch.load(BEST_MODEL_PATH, map_location=device)
num_classes = best_ckpt['num_classes']

print(f"Loaded best model from epoch {best_ckpt['epoch'] + 1}")
print(f"Val accuracy: {best_ckpt['val_acc']:.4f}")
print(f"Num classes:  {num_classes}")

# Rebuild model and load weights
model = train.FontNet(num_classes=num_classes).to(device)
model.load_state_dict(best_ckpt['model_state'])
model.eval()
print("Model ready for export.")

# Export as TorchScript for HF Spaces inference
model_cpu = model.cpu().eval()
scripted  = torch.jit.script(model_cpu)
scripted.save(f'{DRIVE_DIR}/font_model_scripted.pt')

size_mb = os.path.getsize(f'{DRIVE_DIR}/font_model_scripted.pt') / 1e6
print(f"✅ Model exported: {size_mb:.1f} MB")
print(f"✅ Saved to: {DRIVE_DIR}/font_model_scripted.pt")